
# 02 — Estimativa de idade com MiVOLO e UTKFace

## Objetivo

Este notebook realiza a estimativa de idade das imagens da base de dados UTKFace utilizando o modelo MiVOLO, com pesos pré-treinados.

O modelo utiliza um detector YOLOv8 para identificação de pessoas e rostos, associado ao modelo MiVOLO para estimativa de idade e gênero.

O processamento é realizado individualmente para cada imagem, utilizando a configuração experimental definida neste notebook.

Os resultados são registrados em arquivos CSV, contendo a identificação da imagem, a idade de referência extraída do nome do arquivo, a idade estimada, o erro absoluto e o status da execução.

## Organização experimental

A base UTKFace está dividida em três partições (`part1`, `part2` e `part3`), processadas separadamente por meio da mesma configuração experimental.

A partição utilizada é definida por um parâmetro do notebook, permitindo gerar arquivos de resultados independentes para cada lote.

## Modelos utilizados

- Detector: `yolov8x_person_face.pt`
- Modelo de estimativa: `model_imdb_cross_person_4.22_99.46.pth.tar`

Os modelos são utilizados com os pesos pré-treinados, sem treinamento ou ajuste adicional neste experimento.


In [ ]:

# ============================================================
# INSTALAÇÃO DAS DEPENDÊNCIAS E CLONAGEM DO MIVOLO
# ============================================================

import os
import sys

REPOSITORIO_MIVOLO = "https://github.com/WildChlamydia/MiVOLO.git"
DIRETORIO_MIVOLO = "/content/MiVOLO"

# ------------------------------------------------------------
# CLONAGEM DO REPOSITÓRIO
# ------------------------------------------------------------

if not os.path.exists(DIRETORIO_MIVOLO):

    !git clone {REPOSITORIO_MIVOLO} {DIRETORIO_MIVOLO}

else:

    print("Repositório MiVOLO já existe no ambiente.")

# ------------------------------------------------------------
# INSTALAÇÃO DAS DEPENDÊNCIAS
# ------------------------------------------------------------

%cd /content/MiVOLO

!pip install -q -r requirements.txt
!pip install -q .

# Compatibilidade das versões utilizadas no experimento
!pip uninstall -y timm -q
!pip install -q "timm==0.8.13.dev0"
!pip install -q "ultralytics==8.1.0"
!pip install -q transformers
!pip install -q gdown

# ------------------------------------------------------------
# PREPARAÇÃO DO AMBIENTE
# ------------------------------------------------------------

if DIRETORIO_MIVOLO not in sys.path:
    sys.path.append(DIRETORIO_MIVOLO)

PASTA_MODELOS = os.path.join(
    DIRETORIO_MIVOLO,
    "models"
)

os.makedirs(PASTA_MODELOS, exist_ok=True)

print("\nInstalação e configuração inicial concluídas.")
print(f"Repositório: {DIRETORIO_MIVOLO}")
print(f"Pasta dos modelos: {PASTA_MODELOS}")


In [ ]:

# ============================================================
# CONFIGURAÇÃO E CÓPIA DOS MODELOS MiVOLO
# ============================================================

import os
import shutil

PASTA_MODELOS = "/content/MiVOLO/models"

# Diretório dos modelos armazenados no Google Drive.
# Confira se este caminho corresponde à sua pasta.

PASTA_MODELOS_DRIVE = (
    "/content/drive/MyDrive/Classroom/"
    "9923 31 - TRABALHO DE CONCLUSÃO DE CURSO 2026/"
    "DESENVOLVIMENTO/Ambientes Colab/Modelos MiVOLO"
)

MODELO_YOLO = "yolov8x_person_face.pt"
MODELO_MIVOLO = "model_imdb_cross_person_4.22_99.46.pth.tar"

os.makedirs(PASTA_MODELOS, exist_ok=True)

# ------------------------------------------------------------
# CÓPIA DOS MODELOS
# ------------------------------------------------------------

for modelo in [MODELO_YOLO, MODELO_MIVOLO]:

    origem = os.path.join(PASTA_MODELOS_DRIVE, modelo)
    destino = os.path.join(PASTA_MODELOS, modelo)

    if not os.path.isfile(origem):
        raise FileNotFoundError(
            f"Modelo não encontrado no Google Drive:\n{origem}"
        )

    shutil.copy2(origem, destino)

    print(f"Modelo copiado: {modelo}")

# ------------------------------------------------------------
# VERIFICAÇÃO DOS ARQUIVOS
# ------------------------------------------------------------

print("\nVerificação dos modelos:")

for modelo in [MODELO_YOLO, MODELO_MIVOLO]:

    caminho = os.path.join(PASTA_MODELOS, modelo)

    tamanho_mb = os.path.getsize(caminho) / (1024 * 1024)

    print(f"OK: {modelo} — {tamanho_mb:.2f} MB")


In [ ]:

# ============================================================
# VERIFICAÇÃO DAS VERSÕES E DO AMBIENTE
# ============================================================

import torch
import timm
import ultralytics
import transformers

print("=== VERIFICAÇÃO DO AMBIENTE MIVOLO ===")

print(f"PyTorch: {torch.__version__}")
print(f"timm: {timm.__version__}")
print(f"Ultralytics: {ultralytics.__version__}")
print(f"Transformers: {transformers.__version__}")

print(f"\nCUDA disponível: {torch.cuda.is_available()}")

if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
else:
    print("Dispositivo de execução: CPU")


In [ ]:

# ============================================================
# AJUSTE DE COMPATIBILIDADE DO PYTORCH
# ============================================================

import torch

# Evita aplicar o ajuste mais de uma vez.
if not hasattr(torch, "_mivolo_original_load"):

    torch._mivolo_original_load = torch.load

    def torch_load_compat(*args, **kwargs):
        kwargs["weights_only"] = False
        return torch._mivolo_original_load(*args, **kwargs)

    torch.load = torch_load_compat

    print("Ajuste de compatibilidade aplicado.")

else:
    print("Ajuste de compatibilidade já aplicado.")


In [ ]:

# ============================================================
# CARREGAMENTO DO MODELO MIVOLO
# ============================================================

from types import SimpleNamespace
from mivolo.predictor import Predictor

config_obj = SimpleNamespace(
    detector_weights=os.path.join(
        PASTA_MODELOS,
        MODELO_YOLO
    ),
    checkpoint=os.path.join(
        PASTA_MODELOS,
        MODELO_MIVOLO
    ),
    device="cpu",
    with_persons=True,
    disable_faces=False,
    draw=True
)

predictor_mivolo = Predictor(
    config=config_obj,
    verbose=True
)

print("\nMiVOLO carregado com sucesso!")


In [ ]:

# ============================================================
# CONFIGURAÇÃO DA BASE UTKFACE E DOS RESULTADOS
# ============================================================

import os

# Partição que será processada: part1, part2 ou part3
PARTICAO = "part1"

# Diretório principal do UTKFace no Google Drive
PASTA_BASE_UTKFACE = (
    "/content/drive/MyDrive/Classroom/"
    "9923 31 - TRABALHO DE CONCLUSÃO DE CURSO 2026/"
    "DESENVOLVIMENTO/UTKface_inthewild"
)

# Caminho da partição selecionada
PASTA_DATASET = os.path.join(
    PASTA_BASE_UTKFACE,
    PARTICAO
)

# Diretório de armazenamento dos resultados
PASTA_RESULTADOS = (
    "/content/drive/MyDrive/Classroom/"
    "9923 31 - TRABALHO DE CONCLUSÃO DE CURSO 2026/"
    "DESENVOLVIMENTO/Ambientes Colab/resultados/MiVOLO"
)

os.makedirs(PASTA_RESULTADOS, exist_ok=True)

print("=== CONFIGURAÇÃO DO EXPERIMENTO ===")
print(f"Modelo: MiVOLO")
print(f"Base de dados: UTKFace")
print(f"Partição selecionada: {PARTICAO}")
print(f"Diretório do dataset: {PASTA_DATASET}")
print(f"Diretório dos resultados: {PASTA_RESULTADOS}")


In [ ]:

# ============================================================
# VERIFICAÇÃO DO AMBIENTE E DA BASE DE DADOS
# ============================================================

print("=== VERIFICAÇÃO DO AMBIENTE ===")

print("\nDataset:")
if os.path.isdir(PASTA_DATASET):
    print("OK - Diretório encontrado.")
else:
    print("ERRO - Diretório não encontrado.")

print("\nResultados:")
if os.path.isdir(PASTA_RESULTADOS):
    print("OK - Diretório disponível.")
else:
    print("ERRO - Diretório não encontrado.")

print("\nModelos:")

print(
    "YOLO:",
    "OK" if os.path.exists(
        os.path.join(PASTA_MODELOS, MODELO_YOLO)
    ) else "ERRO"
)

print(
    "MiVOLO:",
    "OK" if os.path.exists(
        os.path.join(PASTA_MODELOS, MODELO_MIVOLO)
    ) else "ERRO"
)

EXTENSOES = (".jpg", ".jpeg", ".png")

if os.path.isdir(PASTA_DATASET):
    imagens_particao = [
        f for f in os.listdir(PASTA_DATASET)
        if f.lower().endswith(EXTENSOES)
    ]

    print(f"\nQuantidade de imagens em {PARTICAO}:")
    print(f"{len(imagens_particao)} imagens encontradas.")


In [ ]:
# ============================================================
# TESTE DE INFERÊNCIA COM UMA IMAGEM DO UTKFACE
# ============================================================

import cv2
import matplotlib.pyplot as plt

if not imagens_particao:
    raise ValueError("Nenhuma imagem disponível para o teste.")

nome_teste = imagens_particao[0]
caminho_teste = os.path.join(PASTA_DATASET, nome_teste)

img_teste = cv2.imread(caminho_teste)

if img_teste is None:
    raise ValueError("Não foi possível carregar a imagem de teste.")

detected_objects, out_img = predictor_mivolo.recognize(img_teste)

print("Arquivo:", nome_teste)
print("Idades encontradas:", detected_objects.ages)
print("Gêneros encontrados:", detected_objects.genders)

plt.figure(figsize=(10, 8))
plt.imshow(cv2.cvtColor(out_img, cv2.COLOR_BGR2RGB))
plt.axis("off")
plt.show()

In [ ]:

# ============================================================
# PROCESSAMENTO DAS IMAGENS COM MIVOLO
# ============================================================

import os
import re
import time
import cv2
import pandas as pd

ARQUIVO_RESULTADO = os.path.join(
    PASTA_RESULTADOS,
    f"resultados_mivolo_utkface_{PARTICAO}.csv"
)

CHECKPOINT_A_CADA = 50
EXTENSOES = (".jpg", ".jpeg", ".png")

colunas = [
    "Arquivo",
    "Idade Real",
    "Idade Estimada",
    "Erro Absoluto",
    "Status"
]

# Listar imagens da partição
imagens = sorted([
    nome for nome in os.listdir(PASTA_DATASET)
    if nome.lower().endswith(EXTENSOES)
])

print(f"Total de imagens encontradas: {len(imagens)}")


# ============================================================
# CARREGAR RESULTADOS EXISTENTES
# ============================================================

resultados_por_arquivo = {}

if os.path.exists(ARQUIVO_RESULTADO) and os.path.getsize(ARQUIVO_RESULTADO) > 0:

    df_existente = pd.read_csv(ARQUIVO_RESULTADO)

    # Manter somente as colunas esperadas
    df_existente = df_existente[colunas]

    # Preservar os registros existentes, identificando cada imagem
    for registro in df_existente.to_dict("records"):
        resultados_por_arquivo[str(registro["Arquivo"])] = registro

    # Somente registros OK serão considerados concluídos.
    # Registros com erro serão novamente tentados.
    processados = {
        arquivo
        for arquivo, registro in resultados_por_arquivo.items()
        if registro["Status"] == "OK"
    }

    print("Checkpoint existente carregado.")
    print(f"Imagens já processadas com sucesso: {len(processados)}")

else:
    processados = set()
    print("Nenhum checkpoint encontrado. Iniciando do zero.")


# ============================================================
# PROCESSAMENTO DAS IMAGENS
# ============================================================

inicio_total = time.time()
novos_processados = 0

for i, nome_arquivo in enumerate(imagens, start=1):

    # Pular imagens que já possuem processamento bem-sucedido
    if nome_arquivo in processados:
        continue

    caminho = os.path.join(PASTA_DATASET, nome_arquivo)

    # Inicializar idade para evitar reaproveitamento de valores
    idade_real = None

    try:

        # ----------------------------------------------------
        # EXTRAÇÃO DA IDADE DE REFERÊNCIA
        # ----------------------------------------------------

        match = re.match(r"(\d+)_", nome_arquivo)

        if not match:
            raise ValueError(
                "Não foi possível identificar a idade no nome do arquivo."
            )

        idade_real = int(match.group(1))


        # ----------------------------------------------------
        # LEITURA DA IMAGEM
        # ----------------------------------------------------

        img = cv2.imread(caminho)

        if img is None:
            raise ValueError("Não foi possível carregar a imagem.")


        # ----------------------------------------------------
        # INFERÊNCIA COM MIVOLO
        # ----------------------------------------------------

        detected_objects, _ = predictor_mivolo.recognize(img)

        idades = [
            idade for idade in detected_objects.ages
            if idade is not None
        ]

        if not idades:
            raise ValueError("Nenhuma idade foi estimada.")


        # ----------------------------------------------------
        # REGISTRO DA ESTIMATIVA
        # ----------------------------------------------------

        # A UTKFace apresenta uma pessoa por imagem.
        # Portanto, utiliza-se a primeira idade encontrada.
        idade_estimada = float(idades[0])

        erro_absoluto = abs(idade_real - idade_estimada)

        registro = {
            "Arquivo": nome_arquivo,
            "Idade Real": idade_real,
            "Idade Estimada": round(idade_estimada, 2),
            "Erro Absoluto": round(erro_absoluto, 2),
            "Status": "OK"
        }

        print(
            f"[{i}/{len(imagens)}] "
            f"{nome_arquivo} | "
            f"Referência: {idade_real} | "
            f"Estimativa: {idade_estimada:.2f} | "
            f"Erro: {erro_absoluto:.2f}"
        )


    except Exception as e:

        # Registrar a falha sem interromper o processamento
        registro = {
            "Arquivo": nome_arquivo,
            "Idade Real": idade_real,
            "Idade Estimada": None,
            "Erro Absoluto": None,
            "Status": f"ERRO: {str(e)}"
        }

        print(
            f"[{i}/{len(imagens)}] "
            f"{nome_arquivo} | ERRO: {e}"
        )


    # Inserir ou atualizar o registro da imagem.
    # Assim, uma nova tentativa substitui o erro anterior.
    resultados_por_arquivo[nome_arquivo] = registro

    novos_processados += 1


    # ========================================================
    # SALVAMENTO PERIÓDICO (CHECKPOINT)
    # ========================================================

    if novos_processados >= CHECKPOINT_A_CADA:

        df_checkpoint = pd.DataFrame(
            resultados_por_arquivo.values(),
            columns=colunas
        )

        df_checkpoint.to_csv(
            ARQUIVO_RESULTADO,
            index=False,
            encoding="utf-8-sig"
        )

        print("\n" + "=" * 70)
        print(f"CHECKPOINT SALVO: {len(df_checkpoint)} registros")
        print(f"Arquivo: {ARQUIVO_RESULTADO}")
        print("=" * 70 + "\n")

        novos_processados = 0


# ============================================================
# SALVAMENTO FINAL
# ============================================================

df_final = pd.DataFrame(
    resultados_por_arquivo.values(),
    columns=colunas
)

df_final.to_csv(
    ARQUIVO_RESULTADO,
    index=False,
    encoding="utf-8-sig"
)

tempo_total = time.time() - inicio_total

print("\n" + "=" * 70)
print("PROCESSAMENTO FINALIZADO")
print("=" * 70)
print(f"Total de registros: {len(df_final)}")
print(f"Arquivo salvo: {ARQUIVO_RESULTADO}")
print(f"Tempo desta execução: {tempo_total / 60:.2f} minutos")
print("=" * 70)


In [ ]:

# ============================================================
# RESUMO DOS RESULTADOS DO MIVOLO
# ============================================================

df_sucesso = df_final[
    df_final["Status"] == "OK"
].copy()

df_erro = df_final[
    df_final["Status"] != "OK"
].copy()

print("=== RESUMO DO PROCESSAMENTO MIVOLO ===")
print(f"Partição: {PARTICAO}")
print(f"Total de registros: {len(df_final)}")
print(f"Processamentos bem-sucedidos: {len(df_sucesso)}")
print(f"Processamentos com erro: {len(df_erro)}")

if not df_sucesso.empty:

    mae = df_sucesso["Erro Absoluto"].mean()

    print(f"\nMAE: {mae:.2f} anos")

else:
    print("\nNão há registros bem-sucedidos para calcular o MAE.")

print(f"\nArquivo de resultados: {ARQUIVO_RESULTADO}")
